# Custom Tax Configuration Examples

This notebook demonstrates how to use custom tax configurations for different scenarios.

## What's New

The tax calculator now supports:
- **Built-in configs**: 2024 and 2025 tax years
- **Custom configs**: Create your own tax brackets, rates, and rules
- **What-if analysis**: Compare different policy scenarios

## Import the Package

In [ ]:
from taxcalc import (
    FilingStatus,
    TaxpayerInfo,
    IncomeData,
    ItemizedDeductions,
    TaxCalculator,
    TaxConfig,
    get_tax_config_2024,
    get_tax_config_2025
)

import copy

## Example 1: Compare 2024 vs 2025

Let's see how the same scenario is taxed in 2024 vs 2025.

In [ ]:
# Define your scenario
taxpayer = TaxpayerInfo(
    filing_status=FilingStatus.MARRIED_JOINT,
    age_primary=45,
    age_spouse=43
)

income = IncomeData(
    wages_w2=400000,
    long_term_capital_gains=100000,
    qualified_dividends=15000
)

itemized = ItemizedDeductions(
    state_local_income_taxes=25000,
    mortgage_interest=35000,
    cash_contributions_50pct_orgs=40000
)

# Calculate for 2024
calc_2024 = TaxCalculator(taxpayer, income, itemized, config=get_tax_config_2024())
results_2024 = calc_2024.calculate_all_taxes()

# Calculate for 2025
calc_2025 = TaxCalculator(taxpayer, income, itemized, config=get_tax_config_2025())
results_2025 = calc_2025.calculate_all_taxes()

# Compare
print(f"2024 Total Tax: ${results_2024.total_tax_liability:,.2f}")
print(f"2025 Total Tax: ${results_2025.total_tax_liability:,.2f}")
print(f"Difference:     ${results_2025.total_tax_liability - results_2024.total_tax_liability:,.2f}")
print(f"\nMain change: CA SDI rate increased from {calc_2024.config.ca_sdi_rate*100:.1f}% to {calc_2025.config.ca_sdi_rate*100:.1f}%")

## Example 2: Custom Brackets for 2026

Let's create a hypothetical 2026 configuration with adjusted brackets.

In [ ]:
# Start with 2025 as a base
config_2026 = copy.deepcopy(get_tax_config_2025())
config_2026.tax_year = 2026

# Adjust standard deduction (+3% inflation)
for status in FilingStatus:
    config_2026.standard_deductions[status] = int(
        config_2026.standard_deductions[status] * 1.03
    )

# Adjust federal brackets (+3% inflation)
for status in FilingStatus:
    config_2026.federal_brackets[status] = [
        (int(bracket * 1.03) if bracket != float('inf') else float('inf'), rate)
        for bracket, rate in config_2026.federal_brackets[status]
    ]

# Hypothetical: Increase SALT cap to $15,000
config_2026.salt_cap = 15000

print("2026 Hypothetical Changes:")
print(f"  Standard Deduction (MFJ): ${config_2026.standard_deductions[FilingStatus.MARRIED_JOINT]:,}")
print(f"  SALT Cap: ${config_2026.salt_cap:,}")
print(f"  First bracket (MFJ): ${config_2026.federal_brackets[FilingStatus.MARRIED_JOINT][0][0]:,} at {config_2026.federal_brackets[FilingStatus.MARRIED_JOINT][0][1]*100:.0f}%")

## Example 3: What-If Analysis - Modify Single Parameters

You can easily modify just one parameter to see its impact.

In [ ]:
# Scenario: High-tax state resident with large SALT deduction
taxpayer_high_salt = TaxpayerInfo(filing_status=FilingStatus.MARRIED_JOINT, age_primary=50, age_spouse=48)
income_high_salt = IncomeData(wages_w2=400000)
itemized_high_salt = ItemizedDeductions(
    state_local_income_taxes=60000,  # High state taxes
    real_estate_taxes=20000,
    mortgage_interest=30000
)

# Compare different SALT cap scenarios
salt_scenarios = [
    ("Current ($10K cap)", 10000),
    ("Proposed ($15K cap)", 15000),
    ("Proposed ($20K cap)", 20000),
    ("No cap", float('inf'))
]

print("Impact of Different SALT Caps:\n")
print(f"{'Scenario':<25} {'Total Tax':>15} {'Difference':>15}")
print("-" * 55)

base_tax = None
for name, cap_amount in salt_scenarios:
    config = get_tax_config_2024()
    config.salt_cap = cap_amount
    config.salt_cap_mfs = cap_amount / 2 if cap_amount != float('inf') else float('inf')
    
    calc = TaxCalculator(taxpayer_high_salt, income_high_salt, itemized_high_salt, config=config)
    results = calc.calculate_all_taxes()
    
    if base_tax is None:
        base_tax = results.total_tax_liability
        diff_str = "-"
    else:
        diff = results.total_tax_liability - base_tax
        diff_str = f"${diff:>14,.2f}"
    
    print(f"{name:<25} ${results.total_tax_liability:>14,.2f} {diff_str:>15}")

## Example 4: Build a Completely Custom Tax System

You can build any tax system you can imagine!

In [ ]:
# Create a simple flat tax system
flat_tax_config = TaxConfig(
    tax_year=2024,
    # 20% flat federal tax
    federal_brackets={
        status: [(float('inf'), 0.20)] for status in FilingStatus
    },
    # $50K standard deduction
    standard_deductions={
        status: 50000 for status in FilingStatus
    },
    # No preferential treatment for capital gains
    ltcg_brackets={
        status: [(float('inf'), 0.00)] for status in FilingStatus
    },
    # 10% flat California tax
    ca_brackets={
        status: [(float('inf'), 0.10)] for status in FilingStatus
    },
    ca_standard_deductions={
        status: 50000 for status in FilingStatus
    },
    # Disable AMT and other complex taxes
    amt_exemption={status: 999999999 for status in FilingStatus},
    amt_phaseout_start={status: 999999999 for status in FilingStatus},
    amt_rate_threshold={status: 999999999 for status in FilingStatus},
    niit_threshold={status: 999999999 for status in FilingStatus},
    medicare_threshold={status: 999999999 for status in FilingStatus},
    # No SALT cap
    salt_cap=float('inf'),
    salt_cap_mfs=float('inf'),
    ca_sdi_rate=0.011,
)

# Test it
test_taxpayer = TaxpayerInfo(filing_status=FilingStatus.SINGLE, age_primary=35)
test_income = IncomeData(wages_w2=150000, long_term_capital_gains=50000)
test_itemized = ItemizedDeductions()

calc_current = TaxCalculator(test_taxpayer, test_income, test_itemized, config=get_tax_config_2024())
calc_flat = TaxCalculator(test_taxpayer, test_income, test_itemized, config=flat_tax_config)

results_current = calc_current.calculate_all_taxes()
results_flat = calc_flat.calculate_all_taxes()

print("Flat Tax vs Current System:\n")
print(f"{'Metric':<30} {'Current':>15} {'Flat Tax':>15}")
print("-" * 60)
print(f"{'Gross Income':<30} ${test_income.wages_w2 + test_income.long_term_capital_gains:>14,.0f} ${test_income.wages_w2 + test_income.long_term_capital_gains:>14,.0f}")
print(f"{'Federal Tax':<30} ${results_current.total_federal_tax:>14,.2f} ${results_flat.total_federal_tax:>14,.2f}")
print(f"{'California Tax':<30} ${results_current.total_california_tax:>14,.2f} ${results_flat.total_california_tax:>14,.2f}")
print(f"{'Total Tax':<30} ${results_current.total_tax_liability:>14,.2f} ${results_flat.total_tax_liability:>14,.2f}")
print(f"{'Effective Rate':<30} {results_current.effective_total_rate*100:>14,.2f}% {results_flat.effective_total_rate*100:>14,.2f}%")

## Example 5: Using Custom Configs in Your Own Analysis

Here's a template for your own scenarios:

In [ ]:
# Step 1: Define your taxpayer and income
my_taxpayer = TaxpayerInfo(
    filing_status=FilingStatus.MARRIED_JOINT,
    age_primary=40,
    age_spouse=38
)

my_income = IncomeData(
    wages_w2=250000,
    long_term_capital_gains=50000
)

my_itemized = ItemizedDeductions(
    state_local_income_taxes=20000,
    mortgage_interest=25000,
    cash_contributions_50pct_orgs=15000
)

# Step 2: Choose a config (2024, 2025, or custom)
my_config = get_tax_config_2025()  # or get_tax_config_2024()

# Step 3 (Optional): Modify the config for what-if analysis
# my_config.salt_cap = 15000  # What if SALT cap was $15K?
# my_config.ca_sdi_rate = 0.015  # What if SDI went up to 1.5%?

# Step 4: Calculate
my_calc = TaxCalculator(my_taxpayer, my_income, my_itemized, config=my_config)
my_results = my_calc.calculate_all_taxes()

# Step 5: View results
print(my_calc.generate_summary_report(my_results))

## Summary

You can now:

1. **Use built-in configs**: `get_tax_config_2024()` or `get_tax_config_2025()`
2. **Modify existing configs**: Change specific parameters like SALT cap, SDI rate, etc.
3. **Build custom configs**: Create completely new tax systems
4. **Compare scenarios**: Run the same taxpayer through different configurations

### Common Use Cases:

- **Year-over-year planning**: Compare 2024 vs 2025
- **Policy analysis**: Test proposed tax changes
- **What-if scenarios**: "What if they raise the SALT cap?"
- **Tax reform simulations**: Build hypothetical tax systems

### Next Steps:

- Experiment with different parameters
- Compare multiple scenarios side-by-side
- Use pandas DataFrames to organize comparison results
- Create visualizations of tax differences